# 3 · Seasonality

**Seasonality** is a regular, calendar-driven change in the mean, for example over the week or the year.

| Feature | Best for | How |
|---|---|---|
| **Seasonal indicators** (one-hot of the period) | short seasons with few observations per cycle, e.g. day-of-week | `DeterministicProcess(seasonal=True)` |
| **Fourier features** (sin/cos pairs) | long seasons with many observations, e.g. day-of-year | `CalendarFourier(freq, order)` |

**Process:**
1. **Seasonal plot** to see the pattern.
2. **Periodogram** to choose the Fourier order.
3. **Build the features, fit, and forecast.**
4. **Check** that the deseasonalized periodogram is flat.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plot_params = dict(color="0.75", style=".-", markeredgecolor="0.25", markerfacecolor="0.25", legend=False)

DATA = Path("../input/store-sales-time-series-forecasting")   # change to where you unzipped the data

In [ ]:
import seaborn as sns
from scipy.signal import periodogram


def seasonal_plot(X, y, period, freq, ax=None):
    """One line per `period` (e.g. week), plotted over `freq` (e.g. day of week)."""
    if ax is None:
        _, ax = plt.subplots()
    palette = sns.color_palette("husl", n_colors=X[period].nunique())
    ax = sns.lineplot(x=freq, y=y, hue=period, data=X, errorbar=None, ax=ax, palette=palette, legend=False)
    ax.set_title(f"Seasonal Plot ({period}/{freq})")
    return ax


def plot_periodogram(ts, detrend="linear", ax=None):
    fs = pd.Timedelta("365D") / pd.Timedelta("1D")
    frequencies, spectrum = periodogram(ts, fs=fs, detrend=detrend, window="boxcar", scaling="spectrum")
    if ax is None:
        _, ax = plt.subplots()
    ax.step(frequencies, spectrum, color="purple")
    ax.set_xscale("log")
    ax.set_xticks([1, 2, 4, 6, 12, 26, 52, 104])
    ax.set_xticklabels(["Annual (1)", "Semiannual (2)", "Quarterly (4)", "Bimonthly (6)",
                        "Monthly (12)", "Biweekly (26)", "Weekly (52)", "Semiweekly (104)"], rotation=30)
    ax.set(ylabel="Variance", title="Periodogram")
    return ax

In [ ]:
store_sales = pd.read_csv(
    DATA / "train.csv",
    usecols=["store_nbr", "family", "date", "sales", "onpromotion"],
    dtype={"store_nbr": "category", "family": "category", "sales": "float32", "onpromotion": "uint32"},
    parse_dates=["date"],
)
store_sales["date"] = store_sales.date.dt.to_period("D")
store_sales = store_sales.set_index(["store_nbr", "family", "date"]).sort_index()

average_sales = store_sales.groupby("date").mean()["sales"].loc["2017"]

## 1. Seasonal plot

In [ ]:
X = average_sales.to_frame()
X["week"] = X.index.week         # the period: one line per week
X["day"] = X.index.dayofweek     # the x-axis: position within the week
seasonal_plot(X, y="sales", period="week", freq="day");

## 2. Periodogram
Reading it from left to right: count the frequencies until the power drops off, and use that count as the Fourier `order`. Model a strong
*Weekly* bar with indicators instead.

In [ ]:
plot_periodogram(average_sales);

## 3. Seasonal features → fit → forecast

In [ ]:
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

y = average_sales.copy()

fourier = CalendarFourier(freq="ME", order=4)   # monthly season, 4 sin/cos pairs ("YE" = annual)
dp = DeterministicProcess(
    index=y.index,
    constant=True,               # intercept
    order=1,                     # linear trend
    seasonal=True,               # weekly indicators
    additional_terms=[fourier],  # monthly Fourier features
    drop=True,
)
X = dp.in_sample()

model = LinearRegression(fit_intercept=False).fit(X, y)
y_pred = pd.Series(model.predict(X), index=X.index)

X_fore = dp.out_of_sample(steps=90)
y_fore = pd.Series(model.predict(X_fore), index=X_fore.index)

ax = y.plot(**plot_params, alpha=0.5, title="Trend + seasonal forecast")
y_pred.plot(ax=ax, label="Seasonal")
y_fore.plot(ax=ax, label="Forecast", color="C3")
ax.legend();

## 4. Check: is any seasonality left?
The peaks you modelled should be gone from the residuals.

In [ ]:
y_deseason = y - y_pred

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(10, 7))
plot_periodogram(y, ax=ax1).set_title("Original")
plot_periodogram(y_deseason, ax=ax2).set_title("Deseasonalized");

## 5. Holiday features
One-hot the holidays and join them on date. Every other day is 0.

In [ ]:
holidays_events = pd.read_csv(DATA / "holidays_events.csv", parse_dates=["date"])
holidays_events = holidays_events.set_index("date").to_period("D")

holidays = (
    holidays_events
    .query("locale in ['National', 'Regional']")
    .loc["2017":"2017-08-15", ["description"]]
)
X_holidays = pd.get_dummies(holidays, dtype=float)
X_holidays = X_holidays.groupby(level=0).max()   # two events on one day → one row

X2 = X.join(X_holidays).fillna(0.0)

model = LinearRegression(fit_intercept=False).fit(X2, y)
y_pred = pd.Series(model.predict(X2), index=X2.index)

ax = y.plot(**plot_params, alpha=0.5, title="Seasonal + holidays")
y_pred.plot(ax=ax);

## Store Sales: all 1800 series at once
`LinearRegression` handles a wide target (one column per series), so one fit covers every store × family.

In [ ]:
y = store_sales["sales"].unstack(["store_nbr", "family"]).loc["2017"]

dp = DeterministicProcess(
    index=y.index, constant=True, order=1, seasonal=True,
    additional_terms=[CalendarFourier(freq="ME", order=4)], drop=True,
)
X = dp.in_sample()
X["NewYear"] = X.index.dayofyear == 1

model = LinearRegression(fit_intercept=False).fit(X, y)

X_test = dp.out_of_sample(steps=16)               # test period: 2017-08-16 … 08-31
X_test["NewYear"] = X_test.index.dayofyear == 1
y_submit = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns).clip(0.0)
y_submit.iloc[:5, :5]

In [ ]:
test = pd.read_csv(DATA / "test.csv", dtype={"store_nbr": "category", "family": "category"}, parse_dates=["date"])
test["date"] = test.date.dt.to_period("D")
test = test.set_index(["date", "store_nbr", "family"])

y_submit.index.name = "date"
y_submit = y_submit.stack(["store_nbr", "family"]).rename("sales").to_frame()
y_submit.join(test.id).reindex(columns=["id", "sales"]).to_csv("submission.csv", index=False)

**Takeaways**
- Use indicators for the week and Fourier features for the month or year. The periodogram sets the order.
- These features depend only on the date, so you can forecast any number of steps ahead.